In [ ]:
from string import ascii_lowercase
from matplotlib.cm import ScalarMappable
from matplotlib.ticker import MaxNLocator
import os
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import polars as pl
import polars.selectors as cs
from scipy.stats import linregress, mode as ssmode
from functools import partial
import matplotlib.pyplot as plt
from matplotlib.colors import (
    LinearSegmentedColormap,
    Normalize,
    BoundaryNorm,
    rgb_to_hsv,
    hsv_to_rgb,
)
import colormaps
os.environ["PATH"] = os.environ["PATH"] + ":/opt/homebrew/bin"

from jetutils.definitions import (
    PRETTIER_VARNAME,
    DATADIR,
    DUNCANS_REGIONS_NAMES,
    MONTH_NAMES,
    FIGURES,
    UNITS,
    FACTORS,
    get_region,
    labels_to_mask,
    squarify
)
from jetutils.data import standardize
from jetutils.clustering import timeseries_on_map
from jetutils.jet_finding import (
    to_one_large,
    jet_position_as_da,
    average_jet_categories,
    extract_features,
    gaussian_smooth_func,
    find_all_jets,
    compute_jet_props,
    is_polar_gmix,
    compute_widths,
)
from jetutils.plots import (
    COLORS,
    COLORS_EXT,
    TEXTWIDTH_IN,
    plot_dayofyear_trends,
    plot_seasonal,
    plot_trends,
    Clusterplot,
)
from jetutils.anyspell import make_daily, mask_from_spells_pl, subset_around_onset
from simpsom_dask.simpsom import Simpsom
from simpsom_dask import plots as splots
from simpsom_dask.neighborhoods import Neighborhoods


%load_ext IPython.extensions.autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

basepath = Path(f"{DATADIR}/exp9")

In [ ]:
ALL_TIMES = (
    pl.datetime_range(
        start=pl.datetime(1959, 1, 1),
        end=pl.datetime(2023, 1, 1),
        closed="left",
        interval="6h",
        eager=True,
        time_unit="ms",
    )
    .rename("time")
    .to_frame()
)
summer_filter = ALL_TIMES.filter(pl.col("time").dt.month().is_in([6, 7, 8]))
summer = summer_filter["time"]
summer_daily = summer.filter(summer.dt.hour() == 0)

summer_doy = summer_daily.dt.ordinal_day().unique()
n_bootstraps = 100

jets = pl.read_parquet(basepath.joinpath("jets.parquet"))
phat_jets = to_one_large(jets, int_EDJ_threshold=1.3e8)
props_uncat = pl.read_parquet(basepath.joinpath("props.parquet"))
props_uncat = props_uncat.with_columns(
    cs.exclude("time", "jet ID")
    .cast(pl.Float32())
    .replace((float("-inf"), float("inf"), float("nan")), None)
    .cast(pl.Float32())
)
props = squarify(average_jet_categories(props_uncat), ["time", "jet"])
props_summer = props.filter(pl.col("time").dt.month().is_in([6, 7, 8]))
phat_props = pl.read_parquet(basepath.joinpath("props_full.parquet"))
phat_props = phat_props.with_columns(
    cs.exclude("time", "jet")
    .cast(pl.Float32())
    .replace((float("-inf"), float("inf"), float("nan")), None)
    .cast(pl.Float32())
)
cross_catd_ofile = basepath.joinpath("cross.parquet")

jet_pos_da = jet_position_as_da(jets)
# phat_props_catd = average_jet_categories(phat_props, polar_cutoff=0.5)
# phat_props_catd = phat_props_catd.join(phat_props_catd.rolling("time", period="2d", group_by="jet").agg(**{f"{col}_var": pl.col(col).var() for col in ["mean_lon", "mean_lat", "mean_s", "s_star"]}), on=["time", "jet"])
# phat_props_catd = phat_props_catd.with_columns(cs.numeric().cast(pl.Float32()))

phat_props_summer = summer_filter.join(phat_props, on="time")
phat_props_summer = squarify(phat_props_summer, ["time", "jet"])
phat_props_summer = phat_props_summer.filter(pl.col("time").dt.month().is_in([6, 7, 8]))

# cat

In [ ]:
xys = []
# .filter(pl.col("int") > 1.1e8)

xys = np.array(xys)
fig, axes = plt.subplots(
    3, 4, figsize=(11, 9), constrained_layout=True, sharex="all", sharey="all"
)
axes = axes.ravel()
pair = ["s", "theta", "is_polar"]
cmap = LinearSegmentedColormap.from_list(
    "pinkredpurple", [COLORS[2], COLORS_EXT[8], COLORS[1]]
)
bins = np.linspace(0, 1, 31)
gridsize = 18
for month in range(1, 13):
    ax = axes[month - 1]
    X = extract_features(jets, pair, season=month)
    probas = X[pair[2]]
    center_stj = X.filter(pl.col("is_polar") < 0.3).mean()
    center_edj = X.filter(pl.col("is_polar") > 0.7).mean()
    X1D = X["is_polar"]

    im1 = ax.hexbin(X[pair[0]], X[pair[1]], cmap=colormaps.gray_r, gridsize=gridsize)
    im2 = ax.hexbin(
        X[pair[0]], X[pair[1]], C=probas, cmap=colormaps.gray_r, gridsize=gridsize
    )

    plt.draw()

    offsets1 = np.asarray(list(map(tuple, im1.get_offsets())), dtype="f, f")
    offsets2 = np.asarray(list(map(tuple, im2.get_offsets())), dtype="f, f")
    mask12 = np.isin(offsets1, offsets2)
    colors = cmap(im2.get_array())
    colors = rgb_to_hsv(colors[:, :3])
    min_s, max_s = 0.0, 0.9
    min_v, max_v = 0.9, 1.0
    scaling = np.clip(
        im1.get_array()[mask12] / im1.get_array()[mask12].max() * 1.1, 0, 1
    )
    f = lambda x: np.pow(x, 2)
    colors[:, 1] = min_s + f(scaling ** 1.5) * (max_s - min_s)
    colors[:, 2] = max_v - f(scaling) * (max_v - min_v)
    colors = hsv_to_rgb(colors)
    im2.set_array(None)
    im2.set_facecolor(colors)
    # im2.set_linewidths(0.2)
    im2.set_linewidths(np.clip(2 - 3 * (scaling), 0, 2))
    im2.set_edgecolor("white")
    im2 = ax.add_collection(im2)
    if month > 8:
        label = PRETTIER_VARNAME.get(pair[0], pair[0])
        unit = UNITS.get(pair[0], "$~$")
        ax.set_xlabel(f"{label} [{unit}]")
    if month % 4 == 1:
        label = PRETTIER_VARNAME.get(pair[1], pair[1])
        unit = UNITS.get(pair[1], "$~$")
        ax.set_ylabel(f"{label} [{unit}]")
    if pair[0] in ["lev", "theta"]:
        ax.invert_xaxis()
    elif pair[1] in ["lev", "theta"]:
        ax.invert_yaxis()

    ax.set_title(MONTH_NAMES[month - 1])
    ax.scatter(
        *pl.concat([center_stj, center_edj])[pair[:2]].to_numpy().T,
        facecolor="black",
        edgecolor="white",
        marker="X",
        linewidths=1,
        s=45,
    )
    iax = ax.inset_axes([0.65, 0.14, 0.5, 0.5])
    X1D = np.clip(X1D, 0, 1)
    iax.hist(X1D, bins=bins, alpha=0.5, color="black")
    iax.hist(X1D[probas > 0.5], bins=bins, alpha=1.0, color=COLORS[1])
    iax.hist(X1D[probas < 0.5], bins=bins, alpha=1.0, color=COLORS[2])
    iax.set_xticks([0, 0.5, 1])
    iax.set_yticks([])
    iax.spines[["left", "right", "top"]].set_visible(False)
    iax.set_facecolor((1.0, 1.0, 1.0, 0.0))
    plt.draw()
    # break
fig.savefig(f"{FIGURES}/FeatureBased/gmix_demo.pdf")

In [ ]:
plt.ion()
MYPURPLES = LinearSegmentedColormap.from_list(
    "mypurples", ["#ffffff", COLORS_EXT[4], COLORS_EXT[5]]
)
MYPINKS = LinearSegmentedColormap.from_list(
    "mypinks", ["#ffffff", COLORS_EXT[7], COLORS_EXT[8]]
)

to_plot_1 = []
to_plot_2 = []
for month in range(1, 13):
    this_da = jet_pos_da.sel(time=jet_pos_da.time.dt.month == month)
    to_plot_1.append((this_da < 0.5).mean("time"))
    to_plot_2.append((this_da > 0.5).mean("time"))

In [ ]:
clu = Clusterplot(3, 4, get_region(jet_pos_da), row_height=3.4)
im, kwargs = clu.add_contourf(
    [tp2 * 100 for tp2 in to_plot_2],
    transparify=1,
    levels=7,
    q=0.995,
    cmap=colormaps.gothic_r,
    draw_cbar=False,
    titles=MONTH_NAMES,
)
cbar = clu.fig.colorbar(im, ax=clu.axes, shrink=0.99, pad=0.01)
cbar.ax.set_ylabel(r"Eddy-driven jet occurence [$\%$]", fontsize=13)
im, kwargs = clu.add_contourf(
    [tp1 * 100 for tp1 in to_plot_1],
    transparify=1,
    levels=7,
    q=0.995,
    cmap=colormaps.flamingo_r,
    draw_cbar=False,
)
cbar = clu.fig.colorbar(im, ax=clu.axes, shrink=0.99, pad=0.01)
cbar.ax.set_ylabel(r"Subtropical jet occurence [\%]", fontsize=13)
clu.resize_relative([1.0, 1.06])
clu.fig.savefig(f"{FIGURES}/FeatureBased/is_polar_map_physical.pdf")

# seasonal

In [ ]:
data_vars = [
    "mean_lat",
    "s_star",
    "pers",
    "wavinessDC16",
    "width",
    "double_jet_index",
]
fig = plot_seasonal(
    phat_props,
    data_vars,
    nrows=2,
    ncols=3,
    clear=False,
    folder="Variability",
    suffix="_subset",
    numbering=True,
    save=True,
)

In [ ]:
data_vars = [
    "mean_lon",
    "mean_lat",
    "mean_lev",
    "mean_theta",
    "mean_s",
    "s_star",
    "tilt",
    "wavinessR16",
    "wavinessDC16",
    "wavinessFV15",
    "width",
    "int",
    "int_over_europe",
    "pers",
    "njets",
    "double_jet_index",
]
fig = plot_seasonal(
    phat_props.with_columns(cs.numeric().cast(pl.Float32())),
    data_vars,
    nrows=4,
    ncols=4,
    clear=False,
    folder="Variability",
    numbering=True,
    save=True,
)

# Trends

In [ ]:
data_vars = [
    "mean_lat",
    "s_star",
    "pers",
    "wavinessDC16",
    "width",
    "double_jet_index",
]
for season in ["JJA", "DJF", "MAM", "SON", None]:
    fig = plot_trends(
        phat_props,
        data_vars,
        season,
        n_bootstraps=1000,
        nrows=2,
        ncols=3,
        clear=True,
        folder="Variability",
        suffix="_subset",
        numbering=True,
        save=True,
    )

In [ ]:
data_vars = [
    "mean_lon",
    "mean_lat",
    "mean_lev",
    "mean_theta",
    "mean_s",
    "s_star",
    "tilt",
    "wavinessR16",
    "wavinessDC16",
    "wavinessFV15",
    "width",
    "int",
    "int_over_europe",
    "pers",
    "njets",
    "double_jet_index",
]
for season in ["JJA", "DJF", "MAM", "SON", None]:
    fig = plot_trends(
        phat_props,
        data_vars,
        season,
        n_bootstraps=1000,
        nrows=4,
        ncols=4,
        clear=True,
        folder="Variability",
        suffix="",
        numbering=True,
        save=True,
    )

In [ ]:
data_vars = [
    "mean_lon",
    "mean_lat",
    "mean_lev",
    "mean_theta",
    "mean_s",
    "s_star",
    "tilt",
    "wavinessR16",
    "wavinessDC16",
    "wavinessFV15",
    "width",
    "int",
    "int_over_europe",
    "pers",
    "njets",
    "double_jet_index",
]
for season in ["JJA", "DJF", "MAM", "SON", None]:
    fig = plot_trends(
        phat_props,
        data_vars,
        season,
        n_bootstraps=1000,
        nrows=4,
        ncols=4,
        clear=True,
        std=True,
        folder="Variability",
        suffix="_std",
        numbering=True,
        save=True,
    )

# dayofyear trend

In [ ]:
data_vars = [
    "mean_lat",
    "s_star",
    "pers",
    "wavinessDC16",
    "width",
    "double_jet_index",
]
fig = plot_dayofyear_trends(
    phat_props,
    data_vars,
    n_bootstraps=50,
    win_size=90,
    nrows=2,
    ncols=3,
    clear=False,
    folder="Variability",
    suffix="_subset",
    numbering=True,
    save=True,
)

In [ ]:
data_vars = [
    "mean_lat",
    "s_star",
    "pers",
    "wavinessDC16",
    "width",
    "double_jet_index",
]
fig = plot_dayofyear_trends(
    phat_props,
    data_vars,
    n_bootstraps=50,
    win_size=90,
    nrows=2,
    ncols=3,
    clear=False,
    std=True,
    folder="Variability",
    suffix="_std_subset",
    numbering=True,
    save=True,
)

In [ ]:
data_vars = [
    "mean_lon",
    "mean_lat",
    "mean_lev",
    "mean_theta",
    "mean_s",
    "s_star",
    "tilt",
    "wavinessR16",
    "wavinessDC16",
    "wavinessFV15",
    "width",
    "int",
    "int_over_europe",
    "pers",
    "njets",
    "double_jet_index",
]
fig = plot_dayofyear_trends(
    phat_props,
    data_vars,
    n_bootstraps=50,
    win_size=90,
    nrows=4,
    ncols=4,
    clear=False,
    folder="Variability",
    suffix="",
    numbering=True,
    save=True,
)

In [ ]:
data_vars = [
    "mean_lon",
    "mean_lat",
    "mean_lev",
    "mean_theta",
    "mean_s",
    "s_star",
    "tilt",
    "wavinessR16",
    "wavinessDC16",
    "wavinessFV15",
    "width",
    "int",
    "int_over_europe",
    "pers",
    "njets",
    "double_jet_index",
]
fig = plot_dayofyear_trends(
    phat_props,
    data_vars,
    n_bootstraps=50,
    win_size=90,
    nrows=4,
    ncols=4,
    clear=False,
    folder="Variability",
    suffix="_std",
    std=True,
    numbering=True,
    save=True,
)

# SOM

In [ ]:
path_som = Path(DATADIR, "som_stuff")
kwargs = dict(
    sigma=2, sigmaN=1e-5, init="pca", PBC=True, activation_distance="euclidean"
)
nx = 6
ny = 4
net = Simpsom(
    nx,
    ny,
    **kwargs,
)
net.load_weights(path_som.joinpath("som_6_4_pbc_euclidean.npy"))
centers = xr.open_dataarray(path_som.joinpath("centers_som_6_4_pbc_euclidean.nc"))
labels = xr.open_dataarray(path_som.joinpath("labels_som_6_4_pbc_euclidean.nc"))
net.latest_bmus = labels.values

mask = labels_to_mask(labels)
mask_da = xr.DataArray(
    mask, coords={"time": labels.time.values, "cluster": np.arange(net.n_nodes)}
)
populations = net.compute_populations()
coords = net.neighborhoods.coordinates

yearly = mask_da.resample(time="1YE").sum().values
trends = np.zeros(net.n_nodes)
pvalues = trends.copy()
for k, yearly_ in enumerate(yearly.T):
    trends[k], _, _, pvalues[k], _ = linregress(
        np.arange(yearly.shape[0])[yearly_ != 0], yearly_[yearly_ != 0]
    )
RMSE = xr.open_dataarray(path_som.joinpath("RMSE.nc"))
separatedness = xr.open_dataarray(path_som.joinpath("separatedness.nc"))
majority = xr.open_dataarray(path_som.joinpath("majority_wr.nc"))
no_regime = xr.open_dataarray(path_som.joinpath("no_regime.nc"))

## numbering

In [ ]:
outer_grid, inner_grid, coords, outermask = splots.create_outer_grid(6, 4)
nei = Neighborhoods(6, 4, "hexagons", PBC=True)
cmap = colormaps.cet_l_bmw
cmap = LinearSegmentedColormap.from_list(
    "hoho", cmap(np.linspace(0.45, 1.0, cmap.N // 2))
)
from_ = 5
# distances = nei.distances[from_, outer_grid]
distances = np.full(len(coords), np.nan)
edgecolors = np.full(len(coords), "black", dtype=object)
edgecolors[outermask] = "gray"
alphas = np.ones(len(coords))
alphas[outermask] = 0.05
fig, ax = splots.plot_map(
    coords,
    distances,
    polygons="hexagons",
    draw_cbar=False,
    figsize=(0.5 * TEXTWIDTH_IN, 1.55),
    show=False,
    edgecolors="black",
    cmap=None,
    alphas=alphas,
    linewidths=2,
)
xlims = [
    np.amin(coords[~outermask][:, 0]) - 0.8,
    np.amax(coords[~outermask][:, 0]) + 0.8,
]
ylims = [np.amin(coords[~outermask][:, 1]) - 1, np.amax(coords[~outermask][:, 1]) + 1]
for i, c in enumerate(coords):
    x, y = c
    # textcolor = "white" if distances[i] < 2 else "black"
    fontsize = 8 if outermask[i] else 11
    if x > xlims[0] and x < xlims[-1] and y > ylims[0] and y < ylims[-1]:
        ax.text(
            x,
            y,
            f"${outer_grid.flatten()[i] + 1}$",
            va="center",
            ha="center",
            color="black",
            fontsize=fontsize,
        )
ax.set_xlim(xlims)
ax.set_ylim(ylims)
ax.set_aspect("equal")
# ax.set_title(f"Distance to cluster {to_prettier_order(from_)}", pad=5)
plt.savefig(f"{FIGURES}/Variability/som_numbering.pdf")

## wind comp

In [ ]:
clu = Clusterplot(net.y, net.x, get_region(centers), honeycomb=True, numbering=True)
_ = clu.add_contourf(
    centers,
    cmap=colormaps.cet_l_wyor,
    transparify=4,
    levels=9,
    cbar_kwargs={"label": "Wind speed [m/s]", "pad": 0.01},
)
clu.resize_relative([0.95, 1])
plt.savefig(f"{FIGURES}/Variability/som_realspace.pdf")

In [ ]:
from jetutils.jet_finding import compute_jet_props
ds_center = xr.open_dataset(path_som.joinpath("uvst_som_6_4_pbc_euclidean.nc"))
ds_center = ds_center.load()
n_coarsen = 3
base_s_thresh = 16
alignment_thresh = 0.5
int_thresh_factor = 0.4
hole_size = 6
thresholds = None
smooth_func = partial(gaussian_smooth_func, sigma_lon=1, sigma_lat=0.8)
centers_all_jets = find_all_jets(
    ds_center,
    base_s_thresh=base_s_thresh,
    alignment_thresh=alignment_thresh,
    int_thresh_factor=int_thresh_factor,
    smooth_func=smooth_func,
)
centers_all_jets = is_polar_gmix(
    centers_all_jets, ("lon", "lat", "theta"), n_components=2, n_init=20
)

new_id = []
for clu, jets in centers_all_jets.group_by("cluster", maintain_order=True):
    njets = jets["jet ID"].n_unique()
    for idx, jet in jets.group_by("jet ID", maintain_order=True):
        new_jet_id = jet.with_columns(
            pl.col("jet ID")
            + (pl.col("is_polar") > 0.5)
            .cast(pl.UInt16)
            .diff()
            .abs()
            .fill_null(0)
            .cum_sum()
            .cast(pl.UInt32)
            * njets
        )
        njets = jets["jet ID"].n_unique() - 1 + new_jet_id["jet ID"].n_unique()
        new_id.append(new_jet_id.sort("lon"))
centers_all_jets = pl.concat(new_id)
centers_all_jets = centers_all_jets.filter(pl.len().over("cluster", "jet ID") > 5)

centers_props_uncat = compute_jet_props(centers_all_jets)
centers_props_uncat = centers_props_uncat.join(compute_widths(centers_all_jets, ds_center["s"]), on=["cluster", "jet ID"])
centers_props = squarify(average_jet_categories(centers_props_uncat), ["cluster", "jet"])

centers_all_jets.write_parquet(path_som.joinpath("jets_on_som.parquet"))
centers_props_uncat.write_parquet(path_som.joinpath("props_on_som.parquet"))
centers_props.write_parquet(path_som.joinpath("props_cat_on_som.parquet"))

clu = Clusterplot(net.y, net.x, get_region(ds_center), honeycomb=True, numbering=True)
_ = clu.add_contourf(
    centers,
    cmap=colormaps.cet_l_wyor,
    transparify=1,
    levels=[0, 8, 16, 24, 32, 40],
    cbar_kwargs={"label": "Wind speed [m/s]", "pad": 0.01},
)
for indexer, jet in centers_all_jets.group_by(
    ["cluster", "jet ID"], maintain_order=True
):
    ax = clu.axes[indexer[0]]
    is_polar = jet["is_polar"].mean() >= 0.35
    color = COLORS[2 - int(is_polar)]
    ax.plot(jet["lon"] - clu.central_longitude, jet["lat"], lw=4, color=color)
clu.resize_relative([0.95, 1])
plt.savefig(f"{FIGURES}/Variability/som_realspace_and_wind.pdf")

## fig 6

In [ ]:
titles_wr = [
    "NAO-",
    "AR",
    "AL",
    "SB",
    "No regime",
]

fig, axes = plt.subplots(
    2,
    3,
    figsize=(9, 2 * 1.7),
    constrained_layout=True,
    subplot_kw=dict(aspect="equal"),
)

to_plot_dict = {
    "a) Population [days]": (populations, colormaps.bubblegum_r, True, True, {}),
    "b) Majority weather regime": (
        majority,
        colormaps.bold,
        False,
        True,
        {
            "ticks": np.arange(4),
            "ticklabels": titles_wr[:-1],
            "boundaries": np.arange(-0.5, 4.5),
        },
    ),
    r"c) Proportion of no regime [$\%$]": (
        no_regime * 100,
        colormaps.tempo,
        BoundaryNorm(np.arange(0, 80, 10), colormaps.tempo.N, extend="neither"),
        True,
        {},
    ),
    "d) Pop. trend [day / year]": (
        trends,
        colormaps.cet_d_bwr,
        BoundaryNorm(np.linspace(-0.3, 0.3, 7), colormaps.cet_d_bwr.N, extend="both"),
        False,
        {},
    ),
    "e) RMSE": (RMSE, colormaps.amp, True, True, {}),
    "f) Separatedness": (separatedness, colormaps.amp, True, True, {}),
}
for ax, (title, to_plot) in zip(axes.ravel(), to_plot_dict.items()):
    to_plot, cmap, discretify, numbering, more_cbar_kwargs = to_plot
    if isinstance(discretify, bool):
        discretify = discretify
        norm = None
    else:
        norm = discretify
        discretify = False
    fig, ax = net.plot_on_map(
        to_plot,
        fig=fig,
        ax=ax,
        cmap=cmap,
        norm=norm,
        discretify=discretify,
        numbering=numbering,
        cbar_kwargs={"shrink": 0.9} | more_cbar_kwargs,
    )
    ax.set_title(title)

coords = net.neighborhoods.coordinates
where_signif = np.where(pvalues < 0.05)[0]
signif = net.neighborhoods.coordinates[where_signif]
ax = axes.ravel()[3]
ax.scatter(*signif.T, s=80, c="black", marker="x", linewidths=2.0)
for i, c in enumerate(coords):
    x, y = c
    if i in where_signif:
        continue
    ax.text(x, y, f"${i + 1}$", va="center", ha="center", color="black", fontsize=13)
fig.savefig(f"{FIGURES}/Variability/fig06.pdf")

# rt

In [ ]:
from sklearn.metrics import pairwise_distances
from jetutils.anyspell import get_persistent_spell_times_from_som

distances = pairwise_distances(net.weights)
sigma = np.quantile(distances[distances > 0], 0.1)
spells_above_4 = get_persistent_spell_times_from_som(labels, dists=distances, sigma=sigma, nt_before=0, minlen=16, nojune=False)
spells_any = get_persistent_spell_times_from_som(labels, dists=distances, sigma=sigma, nt_before=0, minlen=1, nojune=False)

spells_per_cluster = spells_above_4.filter(pl.col("relative_index") == 0).group_by("value").len()
spells_per_cluster = pl.Series("value", np.arange(24)).to_frame().join(spells_per_cluster, how="left", on="value").fill_null(0)
cmap = colormaps.bubblegum_r
n_stays = spells_per_cluster["len"].to_numpy()

mean_len_per_cluster = spells_any.group_by(["value", "spell"]).len().group_by("value").agg(pl.col("len").mean())
mean_len_per_cluster = pl.Series("value", np.arange(24)).to_frame().join(mean_len_per_cluster, how="left", on="value")
mean_len = mean_len_per_cluster["len"].to_numpy() / 4

max_len_per_cluster = spells_any.group_by(["value", "spell"]).len().group_by("value").agg(pl.col("len").quantile(0.95))
max_len_per_cluster = pl.Series("value", np.arange(24)).to_frame().join(max_len_per_cluster, how="left", on="value")
max_len = max_len_per_cluster["len"].to_numpy() / 4

plot_on_map = [
    {
        "title": r"\# of long stays ($> 4$ days)",
        "data": n_stays,
        "cmap": colormaps.bubblegum_r,
    },
    {
        "title": r"Mean residence time [day]",
        "data": mean_len,
        "cmap": colormaps.bubblegum_r,
    },
    {
        "title": r"$Q_{95}$ of residence times [day]",
        "data": max_len,
        "cmap": colormaps.bubblegum_r,
    },
]

data_vars_and_jets = [
    ("pers", "STJ", colormaps.flamingo_r),
    ("pers", "EDJ", colormaps.gothic_r),
]

for (varname, jet, cmap) in data_vars_and_jets:
    title = f"{jet} {PRETTIER_VARNAME[varname]} [{UNITS.get(varname, '')}]"
    data = phat_props_summer.filter(pl.col("jet") == jet)[varname]
    data = timeseries_on_map(data, net.latest_bmus)[0]
    dico = {
        "title": title,
        "data": data,
        "cmap": cmap,
    }
    plot_on_map.append(dico)

In [ ]:
from matplotlib.gridspec import GridSpec

fig = plt.figure(figsize=(TEXTWIDTH_IN * 1, 4.4))
gs = GridSpec(3, 4, wspace=0.03, hspace=0.3, bottom=0.02, top=0.98, left=0.02, right=0.98)
axes = np.empty(5, dtype=object)
subplot_kw=dict(aspect="equal")
axes[0] = fig.add_subplot(gs[0, slice(1, 3)], **subplot_kw)
for i in range(1, 5):
    iprim = ((i + 1) % 2) * 2
    axes[i] = fig.add_subplot(gs[1 + (i - 1) // 2, slice(0 + iprim, 2 + iprim)], **subplot_kw)
fig.set_constrained_layout(True)
axes = axes.ravel()
for ax, dico, letter in zip(axes, plot_on_map, "abcdefg"):
    title = dico["title"]
    fig, ax = net.plot_on_map(
        **dico,
        smooth_sigma=0,
        show=False,
        fig=fig,
        ax=ax,
        draw_cbar=True,
        discretify=True,
        linewidths=0,
        cbar_kwargs=dict(shrink=0.8),
        numbering=True,
    )
    ax.set_title(f"{letter}) {title}")
fig.set_tight_layout(False)
fig.savefig(f'{FIGURES}/Variability/som_persistence.pdf')

## pathway

In [ ]:
timestepwise = []
group = []
grouplabels = []
for i, (l, group_) in enumerate(labels.groupby(labels.time.dt.dayofyear).groups.items()):
    group.append(group_)
    month = ssmode(labels.time[labels.time.dt.dayofyear == l].dt.month).mode.item()
    day = ssmode(labels.time[labels.time.dt.dayofyear == l].dt.day).mode.item()
    grouplabel = str(day).zfill(2) + "." + str(month).zfill(2)
    grouplabels.append(grouplabel)
    if i % 7 != 6:
        continue
    group = np.concatenate(group)
    coords = net.neighborhoods.coordinates[labels[group]]
    unique, count = np.unique(labels[group], return_counts=True)
    
    coordsmax = coords.max(axis=0, keepdims=True)
    thetas = coords / coordsmax * 2 * np.pi
    xi, zeta = np.cos(thetas), np.sin(thetas)
    mxi, mzeta = np.mean(xi, axis=0), np.mean(zeta, axis=0)
    com = np.arctan2(-mzeta, -mxi) + np.pi
    com = com / 2 / np.pi * coordsmax
    
    maxdx = net.x
    maxdy = net.y
    dx = np.abs(coords[:, 0] - com[0, 0])
    dy = np.abs(coords[:, 1] - com[0, 1])
    mask_periodic = dx > maxdx
    dx[mask_periodic] = maxdx - dx[mask_periodic]
    mask_periodic = dy > maxdy
    dy[mask_periodic] = maxdy - dy[mask_periodic]
    stdx = np.sqrt(np.sum(dx ** 2) / (len(dx) - 1))
    stdy = np.sqrt(np.sum(dy ** 2) / (len(dy) - 1))
    variab = np.asarray([stdx, stdy])
    grouplabel = f"{grouplabels[0]} - {grouplabels[-1]}"
    timestepwise.append((com.squeeze(), variab.squeeze(), unique, count, grouplabel))
    group = []
    grouplabels = []
    
com = np.asarray([step_[0] for step_ in timestepwise])
com_std = np.asarray([step_[1] for step_ in timestepwise])

fig, axes = plt.subplots(4, 3, figsize=(TEXTWIDTH_IN, 5.), constrained_layout=False, subplot_kw={"aspect": "equal"})
cmap = colormaps.bubblegum_r
max_ = np.quantile([np.amax(timestepwis[3]) for timestepwis in timestepwise], 0.8)
norm = BoundaryNorm(MaxNLocator(6).tick_values(0, max_), cmap.N, extend="max")
im = ScalarMappable(norm, cmap)
coords = net.neighborhoods.coordinates
cbar = fig.colorbar(im, ax=axes)
cbar.ax.set_ylabel("Weekly binned population [time step]")
for i, ax in enumerate(axes.ravel()):
    step = i
    letter = ascii_lowercase[i]
    unique, counts, grouplabel = timestepwise[step][2], timestepwise[step][3], timestepwise[step][4]
    to_plot = np.zeros(net.n_nodes)
    to_plot[unique] = counts
    fig, ax = net.plot_on_map(
        to_plot,
        smooth_sigma=0,
        show=False,
        cmap=cmap,
        norm=norm,
        fig=fig,
        ax=ax,
        draw_cbar=False,
        linewidths=0,
    )
    # ax.errorbar(*com[step], *com_std[step][[1, 0]])
    ax.set_title(f"{letter}) {grouplabel}", pad=-2, fontsize=13)
    
    for i, c in enumerate(coords):
        x, y = c
        color = "white" if to_plot[i] > 100 else "black"
        ax.text(x, y, f'${i + 1}$', va='center', ha='center', color=color, fontsize=9)
fig.set_tight_layout(False)
plt.savefig(f"{FIGURES}/Variability/weekly_pathway.pdf")

## jet pos on SOM

In [ ]:
jets = pl.read_parquet(basepath.joinpath("jets.parquet"))
da = jet_position_as_da(jets.filter(pl.col("time").dt.month().is_in([6, 7, 8])))
mask = mask_da.sel(time=da.time.values).values

In [ ]:
clu = Clusterplot(net.y, net.x, get_region(da), honeycomb=True, numbering=True, row_height=5)
stj_da = da < 0.5
edj_da = da > 0.5
for ax in clu.axes:
    ax.autoscale(False)
    
im1, _ = clu.add_any_contour_from_mask(
    stj_da * 100, mask, transparify=1, levels=6, cmap=colormaps.flamingo_r, draw_cbar=False,
)
im2, _ = clu.add_any_contour_from_mask(
    edj_da * 100, mask, transparify=1, levels=6, cmap=colormaps.dense, draw_cbar=False
)
clu.fig.colorbar(im2, ax=clu.axes, pad=-0.045, label=r"EDJ occurence [\%]")
clu.fig.colorbar(im1, ax=clu.axes, pad=0.01, label=r"STJ occurence [\%]")

for indexer, jet in centers_all_jets.group_by(["cluster", "jet ID"], maintain_order=True):
    ax = clu.axes[indexer[0]]
    is_polar = jet["is_polar"].mean() >= 0.35
    color = "yellow" if is_polar else "cyan"
    lo, la = jet[["lon", "lat"]].to_numpy().T
    ax.plot(lo - clu.central_longitude, la, lw=3, color=color, ls="solid")
    # ax.plot(lo - clu.central_longitude, la, lw=5, color=color, ls="dashed")
    
clu.resize_relative([1.06, 1])
clu.fig.savefig(f'{FIGURES}/Variability/jet_pos_on_som.pdf')

## jet props on SOM

In [ ]:
from jetutils.definitions import infer_direction, squarify
data_vars_and_jets = [
    ("mean_lat", "EDJ"),
    ("s_star", "EDJ"),
    ("wavinessDC16", "EDJ"),
    ("width", "EDJ"),
    ("double_jet_index", "STJ"),
    ("width", "STJ"),
    ("wavinessDC16", "STJ"),
    ("s_star", "STJ"),
    ("mean_lat", "STJ"),
]
fig1, axes1 = plt.subplots(3, 3, figsize=(TEXTWIDTH_IN * 1.3, 2.8 * 2), gridspec_kw=dict(wspace=0.15, hspace=0.15, bottom=0.00, top=0.98), sharex="all", subplot_kw=dict(aspect="equal"), constrained_layout=True)
axes1 = axes1.flatten()
fig2, axes2 = plt.subplots(3, 3, figsize=(TEXTWIDTH_IN * 1.3, 2.8 * 2), gridspec_kw=dict(wspace=0.15, hspace=0.15, bottom=0.00, top=0.98), sharex="all", subplot_kw=dict(aspect="equal"), constrained_layout=True)
axes2 = axes2.flatten()
for letter, (varname, jet), (j, ax1) in zip(ascii_lowercase, data_vars_and_jets, enumerate(axes1)):
    factor = FACTORS.get(varname, 1)
    if factor == 1:
        factor_str = ""
    else:
        factor_str = str(int(np.log10(factor)))
        factor_str = r"$10^{" + factor_str + r"} \times $"
    ax2 = axes2[j]
    to_plot_1 = props_summer.filter(pl.col("jet") == jet)[varname]
    to_plot_1 = timeseries_on_map(to_plot_1, net.latest_bmus)[0]
    to_plot_2 = centers_props.filter(pl.col("jet") == jet)[varname].to_numpy()
    title = f"{letter}) {PRETTIER_VARNAME.get(varname, varname)} [{factor_str}{UNITS.get(varname, '~')}]"
    if varname != "double_jet_index":
        title = f"{title}, {jet}"
        cmap = colormaps.gothic_r if jet == "EDJ" else colormaps.flamingo_r
    else:
        cmap = colormaps.bubblegum_r
    for ax in [ax1, ax2]:
        ax.set_title(title)
    feature = np.concatenate([to_plot_1, to_plot_2])
    symmetric = infer_direction(np.nan_to_num(feature)) == 0
    levels = MaxNLocator(10 if symmetric else 6, symmetric=symmetric).tick_values(np.nanmin(feature), np.nanmax(feature))
    norm = BoundaryNorm(levels, cmap.N)
    cmap.set_bad(color="powderblue")
    fig1, ax1 = net.plot_on_map(
        to_plot_1 / factor,
        smooth_sigma=0,
        show=False,
        fig=fig1,
        ax=ax1,
        draw_cbar=True,
        cmap=cmap,
        discretify=True,
        linewidths=0,
        cbar_kwargs=dict(shrink=0.85),
        numbering=True,
    )
    fig2, ax2 = net.plot_on_map(
        np.ma.masked_invalid(to_plot_2) / factor,
        smooth_sigma=0,
        show=False,
        fig=fig2,
        ax=ax2,
        draw_cbar=True,
        cmap=cmap,
        discretify=True,
        linewidths=0,
        cbar_kwargs=dict(shrink=0.85),
        numbering=True,
    )
fig1.savefig(f'{FIGURES}/Variability/props_projected.pdf')
fig2.savefig(f'{FIGURES}/Variability/props_on_som.pdf')